In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
zip_path = "/content/drive/MyDrive/Sepsis Project/sepsis_dataset.zip"

In [ ]:
!unzip -q "$zip_path" -d /content/sepsis_data

In [ ]:
import os

data_path = "/content/sepsis_data/content/sepsis_dataset"
print("Total files:", len(os.listdir(data_path)))

In [ ]:
!mkdir -p /content/dataset

In [ ]:
!rsync -a /content/sepsis_data/content/sepsis_dataset/ /content/dataset/

In [ ]:
!rm -r /content/sepsis_data

In [ ]:
import os

DATA_PATH = "/content/dataset"
print("Total files:", len(os.listdir(DATA_PATH)))

In [ ]:
sample_file = os.path.join(DATA_PATH, files[0])
print(sample_file)

In [ ]:
import pandas as pd

df = pd.read_csv(sample_file, sep='|')
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_PATH = "/content/dataset"   #folder with .psv files

all_rows = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".psv"):
        file_path = os.path.join(DATA_PATH, file)

        try:
            df = pd.read_csv(file_path, sep='|')

            # -------- HR processing --------
            hr = df['HR'].fillna(method='ffill').fillna(method='bfill')

            features = {}

            # Basic stats
            features['HR_last'] = hr.iloc[-1]
            features['HR_mean'] = hr.mean()
            features['HR_std'] = hr.std()
            features['HR_min'] = hr.min()
            features['HR_max'] = hr.max()

            # Trend
            features['HR_trend'] = hr.iloc[-1] - hr.iloc[0]

            # Differences
            diff = hr.diff().fillna(0)
            features['HR_diff_mean'] = diff.mean()
            features['HR_diff_max'] = diff.max()
            features['HR_diff_min'] = diff.min()

            # Recent behavior
            features['HR_recent_mean'] = hr.tail(6).mean()

            # -------- Metadata --------
            features['Patient_ID'] = file.replace('.psv', '')
            features['SepsisLabel'] = df['SepsisLabel'].max()
            features['num_hours'] = len(df)

            all_rows.append(features)

        except:
            continue

# -------- Create DataFrame --------
final_df = pd.DataFrame(all_rows)

# -------- Save dataset --------
final_df.to_csv("/content/hr_features_dataset.csv", index=False)

# -------- Preview --------
print("Shape:", final_df.shape)
print(final_df.head())

In [ ]:
final_df

In [ ]:
final_df.to_csv("hr_features_dataset.csv", index=False)

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_PATH = "/content/dataset"

all_rows = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".psv"):
        file_path = os.path.join(DATA_PATH, file)

        try:
            df = pd.read_csv(file_path, sep='|')

            features = {}

            # ======================
            # HR FEATURES
            # ======================
            hr = df['HR'].ffill().bfill()

            features['HR_last'] = hr.iloc[-1]
            features['HR_mean'] = hr.mean()
            features['HR_std'] = hr.std()
            features['HR_min'] = hr.min()
            features['HR_max'] = hr.max()
            features['HR_trend'] = hr.iloc[-1] - hr.iloc[0]

            diff_hr = hr.diff().fillna(0)
            features['HR_diff_mean'] = diff_hr.mean()
            features['HR_diff_max'] = diff_hr.max()
            features['HR_diff_min'] = diff_hr.min()

            features['HR_recent_mean'] = hr.tail(6).mean()


            # ======================
            # O2Sat FEATURES
            # ======================
            o2 = df['O2Sat'].ffill().bfill()

            features['O2_last'] = o2.iloc[-1]
            features['O2_mean'] = o2.mean()
            features['O2_std'] = o2.std()
            features['O2_min'] = o2.min()
            features['O2_trend'] = o2.iloc[-1] - o2.iloc[0]

            diff_o2 = o2.diff().fillna(0)
            features['O2_diff_min'] = diff_o2.min()

            features['O2_recent_mean'] = o2.tail(6).mean()

            # fraction of time O2Sat < 92
            features['O2_low_frac'] = (o2 < 92).mean()


            # ======================
            # META + TARGET
            # ======================
            features['Patient_ID'] = file.replace('.psv', '')
            features['SepsisLabel'] = df['SepsisLabel'].max()
            features['num_hours'] = len(df)


            all_rows.append(features)

        except:
            continue


# ======================
# FINAL DATAFRAME
# ======================
final_df = pd.DataFrame(all_rows)

# save file
final_df.to_csv("/content/hr_o2_features.csv", index=False)

print("Shape:", final_df.shape)
print(final_df.head())

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "/content/dataset"

rows = []

total_vals = 0
total_missing = 0

for file in os.listdir(DATA_PATH):
    if not file.endswith(".psv"):
        continue

    fp = os.path.join(DATA_PATH, file)
    try:
        df = pd.read_csv(fp, sep='|', usecols=['Temp'])
    except:
        continue

    temp = df['Temp']
    n = len(temp)
    miss = temp.isna().sum()
    frac = miss / n if n > 0 else np.nan

    # ---- pattern analysis ----
    is_na = temp.isna().values

    # missing at start / end
    miss_start = 0
    for v in is_na:
        if v: miss_start += 1
        else: break

    miss_end = 0
    for v in is_na[::-1]:
        if v: miss_end += 1
        else: break

    # longest consecutive missing block
    longest_gap = 0
    curr = 0
    for v in is_na:
        if v:
            curr += 1
            longest_gap = max(longest_gap, curr)
        else:
            curr = 0

    # number of missing segments
    segments = 0
    in_gap = False
    for v in is_na:
        if v and not in_gap:
            segments += 1
            in_gap = True
        elif not v:
            in_gap = False

    rows.append({
        "Patient_ID": file.replace(".psv",""),
        "num_hours": n,
        "temp_missing_count": miss,
        "temp_missing_frac": frac,
        "miss_start": miss_start,
        "miss_end": miss_end,
        "longest_gap": longest_gap,
        "gap_segments": segments
    })

    total_vals += n
    total_missing += miss

# ---- per-patient table ----
miss_df = pd.DataFrame(rows)

# ---- overall stats ----
overall_missing_frac = total_missing / total_vals

print("Overall Temp missing fraction:", overall_missing_frac)
print(miss_df.describe())

# ---- save for inspection ----
miss_df.to_csv("/content/temp_missingness_report.csv", index=False)

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

DATA_PATH = "/content/dataset"

all_rows = []

def process_file(file_path):
    df = pd.read_csv(file_path, sep='|')

    patient_id = os.path.basename(file_path)

    row = {"patient_id": patient_id}

    # =========================
    # HEART RATE FEATURES
    # =========================
    hr = df['HR']

    row['HR_mean'] = hr.mean()
    row['HR_std'] = hr.std()
    row['HR_min'] = hr.min()
    row['HR_max'] = hr.max()
    row['HR_last'] = hr.iloc[-1]

    # trend (slope)
    if hr.notna().sum() > 1:
        x = np.arange(len(hr))
        mask = hr.notna()
        row['HR_trend'] = np.polyfit(x[mask], hr[mask], 1)[0]
    else:
        row['HR_trend'] = np.nan

    # =========================
    # OXYGEN SATURATION FEATURES
    # =========================
    o2 = df['O2Sat']

    row['O2_mean'] = o2.mean()
    row['O2_std'] = o2.std()
    row['O2_min'] = o2.min()
    row['O2_max'] = o2.max()
    row['O2_last'] = o2.iloc[-1]

    # low oxygen (critical)
    row['O2_below_92_frac'] = (o2 < 92).sum() / len(o2)

    # =========================
    # TEMPERATURE FEATURES
    # =========================
    temp = df['Temp']

    total = len(temp)
    missing = temp.isna().sum()

    # missingness
    row['Temp_missing_frac'] = missing / total

    # valid values
    temp_valid = temp.dropna()

    if len(temp_valid) > 0:
        row['Temp_mean'] = temp_valid.mean()
        row['Temp_std'] = temp_valid.std()
        row['Temp_min'] = temp_valid.min()
        row['Temp_max'] = temp_valid.max()
        row['Temp_last'] = temp_valid.iloc[-1]

        # fever indicator (> 38°C)
        row['Temp_fever_frac'] = (temp_valid > 38).sum() / len(temp_valid)

        # hypothermia (< 36°C)
        row['Temp_low_frac'] = (temp_valid < 36).sum() / len(temp_valid)

        # trend
        if len(temp_valid) > 1:
            x = np.arange(len(temp))
            mask = temp.notna()
            row['Temp_trend'] = np.polyfit(x[mask], temp[mask], 1)[0]
        else:
            row['Temp_trend'] = np.nan
    else:
        # if all missing
        row['Temp_mean'] = np.nan
        row['Temp_std'] = np.nan
        row['Temp_min'] = np.nan
        row['Temp_max'] = np.nan
        row['Temp_last'] = np.nan
        row['Temp_fever_frac'] = np.nan
        row['Temp_low_frac'] = np.nan
        row['Temp_trend'] = np.nan

    # =========================
    # LABEL
    # =========================
    row['SepsisLabel'] = df['SepsisLabel'].max()

    return row


# =========================
# PROCESS ALL FILES
# =========================
files = [os.path.join(DATA_PATH, f) for f in os.listdir(DATA_PATH) if f.endswith('.psv')]

for f in tqdm(files):
    try:
        row = process_file(f)
        all_rows.append(row)
    except:
        continue

final_df = pd.DataFrame(all_rows)

print("Final shape:", final_df.shape)
final_df.head()

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

DATA_PATH = "/content/dataset"

all_rows = []

def process_file(file_path):
    df = pd.read_csv(file_path, sep='|')

    patient_id = os.path.basename(file_path)
    row = {"patient_id": patient_id}

    # =========================
    # HEART RATE
    # =========================
    hr = df['HR']
    row['HR_mean'] = hr.mean()
    row['HR_std'] = hr.std()
    row['HR_min'] = hr.min()
    row['HR_max'] = hr.max()
    row['HR_last'] = hr.dropna().iloc[-1] if hr.notna().any() else np.nan

    if hr.notna().sum() > 1:
        x = np.arange(len(hr))
        mask = hr.notna()
        row['HR_trend'] = np.polyfit(x[mask], hr[mask], 1)[0]
    else:
        row['HR_trend'] = np.nan

    # =========================
    # OXYGEN SATURATION
    # =========================
    o2 = df['O2Sat']
    row['O2_mean'] = o2.mean()
    row['O2_std'] = o2.std()
    row['O2_min'] = o2.min()
    row['O2_max'] = o2.max()
    row['O2_last'] = o2.dropna().iloc[-1] if o2.notna().any() else np.nan
    row['O2_below_92_frac'] = (o2 < 92).sum() / len(o2)

    # =========================
    # TEMPERATURE
    # =========================
    temp = df['Temp']
    total = len(temp)
    missing = temp.isna().sum()

    row['Temp_missing_frac'] = missing / total
    temp_valid = temp.dropna()

    if len(temp_valid) > 0:
        row['Temp_mean'] = temp_valid.mean()
        row['Temp_std'] = temp_valid.std()
        row['Temp_min'] = temp_valid.min()
        row['Temp_max'] = temp_valid.max()
        row['Temp_last'] = temp_valid.iloc[-1]
        row['Temp_fever_frac'] = (temp_valid > 38).sum() / len(temp_valid)
        row['Temp_low_frac'] = (temp_valid < 36).sum() / len(temp_valid)

        if len(temp_valid) > 1:
            x = np.arange(len(temp))
            mask = temp.notna()
            row['Temp_trend'] = np.polyfit(x[mask], temp[mask], 1)[0]
        else:
            row['Temp_trend'] = np.nan
    else:
        row.update({
            'Temp_mean': np.nan,
            'Temp_std': np.nan,
            'Temp_min': np.nan,
            'Temp_max': np.nan,
            'Temp_last': np.nan,
            'Temp_fever_frac': np.nan,
            'Temp_low_frac': np.nan,
            'Temp_trend': np.nan
        })

    # =========================
    # BLOOD PRESSURE (NEW)
    # =========================
    sbp = df['SBP']
    dbp = df['DBP']
    map_ = df['MAP']

    # core
    row['SBP_mean'] = sbp.mean()
    row['DBP_mean'] = dbp.mean()
    row['MAP_mean'] = map_.mean()

    # variability
    row['SBP_std'] = sbp.std()
    row['MAP_std'] = map_.std()

    # extremes
    row['SBP_min'] = sbp.min()
    row['MAP_min'] = map_.min()

    # last
    row['MAP_last'] = map_.dropna().iloc[-1] if map_.notna().any() else np.nan

    # thresholds
    row['MAP_low_frac'] = (map_ < 65).sum() / len(map_)
    row['SBP_low_frac'] = (sbp < 90).sum() / len(sbp)

    # derived
    pulse_pressure = sbp - dbp
    row['PulsePressure_mean'] = pulse_pressure.mean()

    shock_index = hr / sbp
    row['ShockIndex_mean'] = shock_index.replace([np.inf, -np.inf], np.nan).mean()

    # trend
    if map_.notna().sum() > 1:
        x = np.arange(len(map_))
        mask = map_.notna()
        row['MAP_trend'] = np.polyfit(x[mask], map_[mask], 1)[0]
    else:
        row['MAP_trend'] = np.nan

    # =========================
    # LABEL
    # =========================
    row['SepsisLabel'] = df['SepsisLabel'].max()

    return row


# =========================
# RUN ON ALL FILES
# =========================
files = [os.path.join(DATA_PATH, f) for f in os.listdir(DATA_PATH) if f.endswith('.psv')]

for f in tqdm(files):
    try:
        all_rows.append(process_file(f))
    except:
        continue

final_df = pd.DataFrame(all_rows)

print("Final shape:", final_df.shape)
final_df.head()

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

DATA_PATH = "/content/dataset"

# create mapping from patient_id → resp features
resp_features_list = []

for file in tqdm(os.listdir(DATA_PATH)):
    if file.endswith(".psv"):
        file_path = os.path.join(DATA_PATH, file)

        try:
            df = pd.read_csv(file_path, sep='|')
            resp = df['Resp']

            row = {}
            row['patient_id'] = file  # must match existing dataframe

            # features
            row['Resp_mean'] = resp.mean()
            row['Resp_std'] = resp.std()
            row['Resp_max'] = resp.max()
            row['Resp_last'] = resp.dropna().iloc[-1] if resp.notna().any() else np.nan

            valid = resp.dropna()
            row['Resp_high_frac'] = (valid >= 22).sum() / len(valid) if len(valid) > 0 else np.nan

            if resp.notna().sum() > 1:
                x = np.arange(len(resp))
                mask = resp.notna()
                row['Resp_trend'] = np.polyfit(x[mask], resp[mask], 1)[0]
            else:
                row['Resp_trend'] = np.nan

            resp_features_list.append(row)

        except:
            continue

# convert to dataframe
resp_df = pd.DataFrame(resp_features_list)

# =========================
# MERGE WITH EXISTING DATA
# =========================
final_df = final_df.merge(resp_df, on="patient_id", how="left")

print("Updated shape:", final_df.shape)
final_df.head()

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "/content/dataset"

cols = ['EtCO2', 'BaseExcess', 'HCO3']

rows = []

total_counts = {c: 0 for c in cols}
missing_counts = {c: 0 for c in cols}

for file in os.listdir(DATA_PATH):
    if not file.endswith(".psv"):
        continue

    fp = os.path.join(DATA_PATH, file)
    try:
        df = pd.read_csv(fp, sep='|', usecols=cols)
    except:
        continue

    row = {"patient_id": file.replace(".psv",""), "num_hours": len(df)}

    for c in cols:
        s = df[c]
        n = len(s)
        miss = s.isna().sum()

        frac = miss / n if n > 0 else np.nan
        row[f"{c}_missing_frac"] = frac

        total_counts[c] += n
        missing_counts[c] += miss

    rows.append(row)

# per-patient dataframe
miss_df = pd.DataFrame(rows)

# overall missing fractions
overall = {c: (missing_counts[c] / total_counts[c]) for c in cols}

print("=== Overall Missing Fractions ===")
for c, v in overall.items():
    print(f"{c}: {v:.4f}")

print("\n=== Per-patient summary ===")
print(miss_df[[f"{c}_missing_frac" for c in cols]].describe())

# save detailed report
miss_df.to_csv("/content/etco2_baseexcess_hco3_missingness.csv", index=False)
print("\nSaved detailed report to /content/etco2_baseexcess_hco3_missingness.csv")

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "/content/dataset"

cols = ['FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN',
        'Alkalinephos', 'Calcium', 'Chloride']

rows = []

total_counts = {c: 0 for c in cols}
missing_counts = {c: 0 for c in cols}

for file in os.listdir(DATA_PATH):
    if not file.endswith(".psv"):
        continue

    fp = os.path.join(DATA_PATH, file)

    try:
        df = pd.read_csv(fp, sep='|', usecols=cols)
    except:
        continue

    row = {
        "patient_id": file.replace(".psv", ""),
        "num_hours": len(df)
    }

    for c in cols:
        s = df[c]
        n = len(s)
        miss = s.isna().sum()

        frac = miss / n if n > 0 else np.nan
        row[f"{c}_missing_frac"] = frac

        total_counts[c] += n
        missing_counts[c] += miss

    rows.append(row)

# create dataframe
miss_df = pd.DataFrame(rows)

# overall missing %
overall = {c: (missing_counts[c] / total_counts[c]) for c in cols}

print("=== Overall Missing Fractions ===")
for c, v in overall.items():
    print(f"{c}: {v:.4f}")

print("\n=== Per-patient summary ===")
print(miss_df[[f"{c}_missing_frac" for c in cols]].describe())

# save report
miss_df.to_csv("/content/lab_missingness_report.csv", index=False)

print("\nSaved report to: /content/lab_missingness_report.csv")

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "/content/dataset"

cols = ['Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate',
        'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total',
        'TroponinI', 'Hct', 'Hgb', 'PTT']

rows = []

total_counts = {c: 0 for c in cols}
missing_counts = {c: 0 for c in cols}

for file in os.listdir(DATA_PATH):
    if not file.endswith(".psv"):
        continue

    fp = os.path.join(DATA_PATH, file)

    try:
        df = pd.read_csv(fp, sep='|', usecols=cols)
    except:
        continue

    row = {
        "patient_id": file.replace(".psv", ""),
        "num_hours": len(df)
    }

    for c in cols:
        s = df[c]
        n = len(s)
        miss = s.isna().sum()

        frac = miss / n if n > 0 else np.nan
        row[f"{c}_missing_frac"] = frac

        total_counts[c] += n
        missing_counts[c] += miss

    rows.append(row)

# create dataframe
miss_df = pd.DataFrame(rows)

# overall missing %
overall = {c: (missing_counts[c] / total_counts[c]) for c in cols}

print("=== Overall Missing Fractions ===")
for c, v in overall.items():
    print(f"{c}: {v:.4f}")

print("\n=== Per-patient summary ===")
print(miss_df[[f"{c}_missing_frac" for c in cols]].describe())

# save report
miss_df.to_csv("/content/lab2_missingness_report.csv", index=False)

print("\nSaved report to: /content/lab2_missingness_report.csv")

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

DATA_PATH = "/content/dataset"

lab_rows = []

for file in tqdm(os.listdir(DATA_PATH)):
    if not file.endswith(".psv"):
        continue

    path = os.path.join(DATA_PATH, file)

    try:
        df = pd.read_csv(path, sep='|', usecols=['Glucose', 'Creatinine'])

        row = {}
        row['patient_id'] = file  # must match existing final_df

        # ---------- Glucose ----------
        g = df['Glucose']
        valid_g = g.dropna()

        row['Glucose_mean'] = valid_g.mean()
        row['Glucose_last'] = valid_g.iloc[-1] if len(valid_g) > 0 else np.nan
        row['Glucose_missing_frac'] = g.isna().mean()

        # ---------- Creatinine ----------
        c = df['Creatinine']
        valid_c = c.dropna()

        row['Creatinine_mean'] = valid_c.mean()
        row['Creatinine_last'] = valid_c.iloc[-1] if len(valid_c) > 0 else np.nan
        row['Creatinine_missing_frac'] = c.isna().mean()

        lab_rows.append(row)

    except:
        continue

# convert to dataframe
lab_df = pd.DataFrame(lab_rows)

# 🔗 merge into your existing dataframe
final_df = final_df.merge(lab_df, on="patient_id", how="left")

print("Updated shape:", final_df.shape)
final_df.head()

In [ ]:
final_df = final_df.drop(columns=['Creatinine_mean'])

In [ ]:
final_df

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

DATA_PATH = "/content/dataset"

cols = [
    'WBC', 'Fibrinogen', 'Platelets',
    'Age', 'Gender', 'Unit1', 'Unit2',
    'HospAdmTime', 'ICULOS'
]

missing_counts = {c: 0 for c in cols}
total_counts = {c: 0 for c in cols}

for file in tqdm(os.listdir(DATA_PATH)):
    if not file.endswith(".psv"):
        continue

    path = os.path.join(DATA_PATH, file)

    try:
        df = pd.read_csv(path, sep='|', usecols=cols)

        for c in cols:
            total_counts[c] += len(df)
            missing_counts[c] += df[c].isna().sum()

    except:
        continue

print("=== Missing Percentage ===")
for c in cols:
    missing_frac = missing_counts[c] / total_counts[c]
    print(f"{c}: {missing_frac:.4f}")

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

DATA_PATH = "/content/dataset"

extra_rows = []

for file in tqdm(os.listdir(DATA_PATH)):
    if file.endswith(".psv"):
        file_path = os.path.join(DATA_PATH, file)

        try:
            df = pd.read_csv(file_path, sep='|')

            row = {}
            row['patient_id'] = file   # must match your existing final_df

            # =========================
            # WBC
            # =========================
            wbc = df['WBC']
            row['WBC_last'] = wbc.dropna().iloc[-1] if wbc.notna().any() else np.nan
            row['WBC_measured'] = 1 if wbc.notna().any() else 0

            valid_wbc = wbc.dropna()
            row['WBC_abnormal'] = ((valid_wbc > 12) | (valid_wbc < 4)).sum() / len(valid_wbc) if len(valid_wbc) > 0 else np.nan

            # =========================
            # Platelets
            # =========================
            plt = df['Platelets']
            row['Platelets_last'] = plt.dropna().iloc[-1] if plt.notna().any() else np.nan
            row['Platelets_measured'] = 1 if plt.notna().any() else 0

            valid_plt = plt.dropna()
            row['Platelets_low_frac'] = (valid_plt < 150).sum() / len(valid_plt) if len(valid_plt) > 0 else np.nan

            # =========================
            # Fibrinogen
            # =========================
            fib = df['Fibrinogen']
            row['Fibrinogen_last'] = fib.dropna().iloc[-1] if fib.notna().any() else np.nan
            row['Fibrinogen_measured'] = 1 if fib.notna().any() else 0

            # =========================
            # Age + Group
            # =========================
            age = df['Age'].iloc[0]
            row['Age'] = age

            if age < 40:
                row['Age_group'] = 0
            elif age < 65:
                row['Age_group'] = 1
            else:
                row['Age_group'] = 2

            # =========================
            # Gender
            # =========================
            row['Gender'] = df['Gender'].iloc[0]

            # =========================
            # Unit
            # =========================
            row['Unit1'] = df['Unit1'].iloc[0]
            row['Unit2'] = df['Unit2'].iloc[0]

            # =========================
            # ICULOS
            # =========================
            iculos = df['ICULOS']
            last_iculos = iculos.iloc[-1]

            row['ICULOS_last'] = last_iculos

            if last_iculos < 24:
                row['ICULOS_bucket'] = 0
            elif last_iculos < 72:
                row['ICULOS_bucket'] = 1
            else:
                row['ICULOS_bucket'] = 2

            extra_rows.append(row)

        except:
            continue


# convert to dataframe
extra_df = pd.DataFrame(extra_rows)

# =========================
# MERGE INTO EXISTING DATA
# =========================
final_df = final_df.merge(extra_df, on="patient_id", how="left")

print("Updated shape:", final_df.shape)
final_df.head()

In [ ]:
final_df.to_csv("/content/final_dataset1.csv", index=False)

In [ ]:
from google.colab import drive
import pandas as pd

# mount drive
drive.mount('/content/drive')

# path to your file (change name if needed)
file_path = "/content/drive/MyDrive/final_dataset1.csv"

# load dataset
df = pd.read_csv(file_path)

# check
print(df.shape)
df.head()

In [ ]:
df.columns

In [ ]:
cols = ['patient_id', 'HR_mean', 'HR_std', 'HR_min', 'HR_max', 'HR_last', 'HR_trend']

# count of missing values
missing_count = df[cols].isna().sum()

# percentage of missing values
missing_percent = (df[cols].isna().mean() * 100)

# combine in one table
missing_df = pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_%': missing_percent
})

print(missing_df)

In [ ]:
cols = ['HR_mean', 'HR_std', 'HR_min', 'HR_max', 'HR_last', 'HR_trend']

for col in cols:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
 cols = ['O2_mean', 'O2_std', 'O2_min', 'O2_max', 'O2_last', 'O2_below_92_frac']

# count
missing_count = df[cols].isna().sum()

# percentage
missing_percent = df[cols].isna().mean() * 100

# combine
missing_df = pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_%': missing_percent
})

print(missing_df)

In [ ]:
cols = ['O2_mean', 'O2_std', 'O2_min', 'O2_max', 'O2_last']

for col in cols:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
cols = [
    'Temp_missing_frac', 'Temp_mean', 'Temp_std',
    'Temp_min', 'Temp_max', 'Temp_last',
    'Temp_fever_frac', 'Temp_low_frac', 'Temp_trend'
]

missing_count = df[cols].isna().sum()
missing_percent = df[cols].isna().mean() * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_%': missing_percent
})

print(missing_df)

In [ ]:
cols_median = [
    'Temp_mean', 'Temp_std',
    'Temp_min', 'Temp_max', 'Temp_last',
    'Temp_trend'
]

for col in cols_median:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
df['Temp_fever_frac'].fillna(0, inplace=True)
df['Temp_low_frac'].fillna(0, inplace=True)

In [ ]:
cols = [
    'SBP_mean', 'DBP_mean', 'MAP_mean',
    'SBP_std', 'MAP_std',
    'SBP_min', 'MAP_min', 'MAP_last',
    'MAP_low_frac', 'SBP_low_frac',
    'PulsePressure_mean', 'ShockIndex_mean',
    'MAP_trend'
]

missing_count = df[cols].isna().sum()
missing_percent = df[cols].isna().mean() * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_%': missing_percent
})

print(missing_df)

In [ ]:
cols_median = [
    'SBP_mean', 'MAP_mean',
    'SBP_std', 'MAP_std',
    'SBP_min', 'MAP_min', 'MAP_last',
    'MAP_trend', 'ShockIndex_mean'
]

for col in cols_median:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
df['DBP_mean'].fillna(df['DBP_mean'].median(), inplace=True)

In [ ]:
# fill missing with median
df['PulsePressure_mean'].fillna(df['PulsePressure_mean'].median(), inplace=True)

In [ ]:
cols = [
    'Resp_mean', 'Resp_std', 'Resp_max',
    'Resp_last', 'Resp_high_frac', 'Resp_trend'
]

missing_count = df[cols].isna().sum()
missing_percent = df[cols].isna().mean() * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_%': missing_percent
})

print(missing_df)

In [ ]:
cols_median = [
    'Resp_mean', 'Resp_std',
    'Resp_max', 'Resp_last',
    'Resp_trend'
]

for col in cols_median:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
df['Resp_high_frac'].fillna(0, inplace=True)

In [ ]:
cols = [
    'Glucose_mean', 'Glucose_last', 'Glucose_missing_frac',
    'Creatinine_last', 'Creatinine_missing_frac'
]

missing_count = df[cols].isna().sum()
missing_percent = df[cols].isna().mean() * 100

print(pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_%': missing_percent
}))

In [ ]:
df['Glucose_mean'].fillna(df['Glucose_mean'].median(), inplace=True)
df['Glucose_last'].fillna(df['Glucose_last'].median(), inplace=True)

df['Creatinine_last'].fillna(df['Creatinine_last'].median(), inplace=True)

In [ ]:
cols = [
    'WBC_last', 'WBC_measured', 'WBC_abnormal',
    'Platelets_last', 'Platelets_measured',
    'Platelets_low_frac',
    'Fibrinogen_last', 'Fibrinogen_measured'
]

missing_count = df[cols].isna().sum()
missing_percent = df[cols].isna().mean() * 100

print(pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_%': missing_percent
}))

In [ ]:
df['WBC_last'].fillna(df['WBC_last'].median(), inplace=True)
df['WBC_abnormal'].fillna(0, inplace=True)

df['Platelets_last'].fillna(df['Platelets_last'].median(), inplace=True)
df['Platelets_low_frac'].fillna(0, inplace=True)

In [ ]:
df.drop(columns=['Fibrinogen_last'], inplace=True)

In [ ]:
cols = [
    'Age', 'Age_group', 'Gender',
    'Unit1', 'Unit2',
    'ICULOS_last', 'ICULOS_bucket'
]

missing_count = df[cols].isna().sum()
missing_percent = df[cols].isna().mean() * 100

print(pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_%': missing_percent
}))

In [ ]:
df['Unit1'].fillna(0, inplace=True)
df['Unit2'].fillna(0, inplace=True)

In [ ]:
df.to_csv("/content/final_cleaned_dataset.csv", index=False)

In [ ]:
from google.colab import drive
import pandas as pd

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Path to your saved file
file_path = "/content/drive/MyDrive/final_cleaned_dataset.csv"

# Step 3: Load dataset
df = pd.read_csv(file_path)

# Step 4: Verify
print("Shape:", df.shape)
df.head()

In [ ]:
# count values
counts = df['SepsisLabel'].value_counts()

print("Non-Sepsis (0):", counts.get(0, 0))
print("Sepsis (1):", counts.get(1, 0))

In [ ]:
percent = df['SepsisLabel'].value_counts(normalize=True) * 100

print("\nPercentage:")
print(percent)

In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: separate features and target
X = df.drop(['SepsisLabel', 'patient_id'], axis=1)
y = df['SepsisLabel']

# Step 2: first split (train + temp)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,   # 30% goes to temp
    random_state=42,
    stratify=y
)

# Step 3: split temp into validation + test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,   # split 30% → 15% val, 15% test
    random_state=42,
    stratify=y_temp
)

# Step 4: check sizes
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# Step 5: check class balance
print("\nTrain distribution:\n", y_train.value_counts(normalize=True))
print("\nValidation distribution:\n", y_val.value_counts(normalize=True))
print("\nTest distribution:\n", y_test.value_counts(normalize=True))

In [ ]:
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

# ======================
# STEP 1: Ensure numpy format (safe for both df & array)
# ======================
X_train_np = np.array(X_train)
X_val_np   = np.array(X_val)
X_test_np  = np.array(X_test)

y_train_np = np.array(y_train)
y_val_np   = np.array(y_val)
y_test_np  = np.array(y_test)

# ======================
# STEP 2: Handle imbalance
# ======================
scale_pos_weight = (y_train_np == 0).sum() / (y_train_np == 1).sum()

# ======================
# STEP 3: Define model
# ======================
model = XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    eval_metric="auc"
)

# ======================
# STEP 4: Train (early stopping)
# ======================
model.fit(
    X_train_np, y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    early_stopping_rounds=50,
    verbose=True
)

# ======================
# STEP 5: Validation performance
# ======================
y_val_prob = model.predict_proba(X_val_np)[:, 1]
val_auc = roc_auc_score(y_val_np, y_val_prob)
print("\nValidation ROC-AUC:", val_auc)

# ======================
# STEP 6: Test performance (FINAL)
# ======================
y_test_prob = model.predict_proba(X_test_np)[:, 1]
test_auc = roc_auc_score(y_test_np, y_test_prob)
print("\nTest ROC-AUC:", test_auc)

# ======================
# STEP 7: Threshold tuning
# ======================
thresholds = [0.5, 0.4, 0.3, 0.25, 0.2]

best_threshold = 0.5
best_recall = 0

print("\nThreshold Tuning:")

for t in thresholds:
    y_pred = (y_test_prob > t).astype(int)

    report = classification_report(y_test_np, y_pred, output_dict=True)
    recall = report['1']['recall']

    print(f"\nThreshold: {t}")
    print(classification_report(y_test_np, y_pred))

    if recall > best_recall:
        best_recall = recall
        best_threshold = t

print("\nBest Threshold (based on recall):", best_threshold)

# ======================
# STEP 8: Save model + threshold
# ======================
save_path = "/content/drive/MyDrive/best_sepsis_model.pkl"

joblib.dump({
    "model": model,
    "threshold": best_threshold
}, save_path)

print("\nModel + threshold saved at:", save_path)

In [ ]:
print("f58 is:", X.columns[58])
print("f57 is:", X.columns[57])

In [1]:
## Identified data leakage features removed before training
leakage_cols = [
    'ICULOS_last',    # col 59 — total ICU stay, encodes outcome
    'ICULOS_bucket',  # col 60 — same thing, just binned
    'Age_group',      # col 55 — redundant, derived directly from Age
]

## Binned version of ICU stay; excluded to avoid redundancy
exclude = ['patient_id', 'SepsisLabel']

In [ ]:
from google.colab import drive
import pandas as pd
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/final_cleaned_dataset.csv")

# Remove leakage + identifiers
drop_cols = ['patient_id', 'SepsisLabel', 'ICULOS_last', 'ICULOS_bucket', 'Age_group']

X = df.drop(columns=drop_cols)
y = df['SepsisLabel']

feature_names = X.columns.tolist()
print(f"Features : {len(feature_names)}")  # should be 55
print(f"Patients : {len(df)}")
print(f"Sepsis % : {y.mean()*100:.1f}%")

# Stratified split — 70 / 15 / 15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"\nTrain : {X_train.shape[0]:>5}  |  Sepsis: {y_train.mean()*100:.1f}%")
print(f"Val   : {X_val.shape[0]:>5}  |  Sepsis: {y_val.mean()*100:.1f}%")
print(f"Test  : {X_test.shape[0]:>5}  |  Sepsis: {y_test.mean()*100:.1f}%")

In [ ]:
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
import matplotlib.pyplot as plt

X_train_np = np.array(X_train)
X_val_np   = np.array(X_val)
X_test_np  = np.array(X_test)

y_train_np = np.array(y_train).ravel()
y_val_np   = np.array(y_val).ravel()
y_test_np  = np.array(y_test).ravel()

neg = (y_train_np == 0).sum()
pos = (y_train_np == 1).sum()
scale_pos_weight = neg / pos
print(f"Class ratio: {scale_pos_weight:.1f}:1")

model = XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    eval_metric="aucpr",
)

model.fit(
    X_train_np, y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    verbose=100,
)

print(f"\nBest iteration : {model.best_iteration}")

y_val_prob  = model.predict_proba(X_val_np)[:, 1]
y_test_prob = model.predict_proba(X_test_np)[:, 1]

val_auc  = roc_auc_score(y_val_np,  y_val_prob)
test_auc = roc_auc_score(y_test_np, y_test_prob)
print(f"Validation ROC-AUC : {val_auc:.4f}")
print(f"Test       ROC-AUC : {test_auc:.4f}")

# F2 threshold
precisions, recalls, pr_thresholds = precision_recall_curve(y_test_np, y_test_prob)
beta     = 2
f2       = (1 + beta**2) * (precisions * recalls) / ((beta**2 * precisions) + recalls + 1e-9)
best_idx = np.argmax(f2)
best_threshold = float(pr_thresholds[best_idx])

print(f"\nBest threshold : {best_threshold:.4f}")
print(f"F2 score       : {f2[best_idx]:.4f}")
print(f"\nFull report at t={best_threshold:.2f}:")
print(classification_report(y_test_np, (y_test_prob > best_threshold).astype(int), zero_division=0))

# PR curve
plt.figure(figsize=(8, 5))
plt.plot(recalls, precisions, color='steelblue', lw=2, label='PR Curve')
plt.scatter(recalls[best_idx], precisions[best_idx],
            marker='*', s=200, color='red', zorder=5,
            label=f'Best t={best_threshold:.2f}')
plt.xlabel('Recall (Sensitivity)')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Early Sepsis Detection')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Feature importance with real names
importances = model.feature_importances_
top_n = 20
idx = np.argsort(importances)[-top_n:][::-1]

plt.figure(figsize=(10, 7))
plt.barh(range(top_n), importances[idx][::-1], color='steelblue')
plt.yticks(range(top_n), [feature_names[i] for i in idx][::-1])
plt.xlabel('Importance Score')
plt.title('Top 20 Features — XGBoost Sepsis Model (No Leakage)')
plt.tight_layout(); plt.show()

# Save
save_path = "/content/drive/MyDrive/best_sepsis_model.pkl"
joblib.dump({
    "model"         : model,
    "threshold"     : best_threshold,
    "feature_names" : feature_names,
    "val_auc"       : val_auc,
    "test_auc"      : test_auc,
}, save_path)
print(f"\nSaved → {save_path}")

In [ ]:
print(df.groupby('Unit2')['SepsisLabel'].agg(['mean', 'count']))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import joblib
import numpy as np
import pandas as pd
import io

# ======================
# LOAD MODEL
# ======================
saved     = joblib.load("/content/drive/MyDrive/best_sepsis_model.pkl")
model     = saved['model']
threshold = saved['threshold']
features  = saved['feature_names']


# ======================
# COLUMN NORMALIZATION (FIXES 'HR' ERROR)
# ======================
def normalize_columns(df):
    df.columns = df.columns.str.strip().str.lower()

    mapping = {
        'hr': 'hr', 'heartrate': 'hr',
        'o2sat': 'o2sat', 'spo2': 'o2sat',
        'temp': 'temp', 'temperature': 'temp',
        'sbp': 'sbp',
        'dbp': 'dbp',
        'map': 'map',
        'resp': 'resp', 'respiratoryrate': 'resp',
        'age': 'age',
        'gender': 'gender',
        'unit1': 'unit1',
        'unit2': 'unit2'
    }

    new_cols = {}
    for col in df.columns:
        key = col.replace(" ", "")
        if key in mapping:
            new_cols[col] = mapping[key]

    df = df.rename(columns=new_cols)
    return df


# ======================
# SAFE FEATURE EXTRACTION
# ======================
def safe_series(df, col):
    return df[col] if col in df.columns else pd.Series([np.nan]*len(df))


def extract_features(df):
    row = {}

    def stat(series, prefix):
        valid = series.dropna()
        row[f'{prefix}_missing_frac'] = series.isna().mean()

        if len(valid) == 0:
            for k in ['mean','std','min','max','last','trend']:
                row[f'{prefix}_{k}'] = 0.0
            return

        row[f'{prefix}_mean']  = valid.mean()
        row[f'{prefix}_std']   = valid.std() if len(valid) > 1 else 0.0
        row[f'{prefix}_min']   = valid.min()
        row[f'{prefix}_max']   = valid.max()
        row[f'{prefix}_last']  = valid.iloc[-1]

        if len(valid) > 1:
            x = np.arange(len(series))
            mask = series.notna()
            row[f'{prefix}_trend'] = np.polyfit(x[mask], series[mask], 1)[0]
        else:
            row[f'{prefix}_trend'] = 0.0

    # vitals
    stat(safe_series(df, 'hr'), 'HR')
    stat(safe_series(df, 'o2sat'), 'O2')
    stat(safe_series(df, 'temp'), 'Temp')
    stat(safe_series(df, 'sbp'), 'SBP')
    stat(safe_series(df, 'dbp'), 'DBP')
    stat(safe_series(df, 'map'), 'MAP')
    stat(safe_series(df, 'resp'), 'Resp')

    # derived
    o2 = safe_series(df, 'o2sat')
    mapv = safe_series(df, 'map')
    sbp = safe_series(df, 'sbp')
    hr  = safe_series(df, 'hr')

    row['O2_below_92_frac'] = (o2 < 92).mean()
    row['MAP_low_frac'] = (mapv < 65).mean()
    row['SBP_low_frac'] = (sbp < 90).mean()
    row['ShockIndex_mean'] = (hr / sbp.replace(0, np.nan)).mean()

    # demographics (safe)
    row['Age'] = df['age'].iloc[0] if 'age' in df.columns else 0
    row['Gender'] = df['gender'].iloc[0] if 'gender' in df.columns else 0
    row['Unit1'] = df['unit1'].iloc[0] if 'unit1' in df.columns else 0
    row['Unit2'] = df['unit2'].iloc[0] if 'unit2' in df.columns else 0

    return row


# ======================
# UI
# ======================
display(HTML("""
<div style='background:#1a1a2e;padding:16px 24px;border-radius:12px;margin-bottom:16px'>
  <h2 style='color:#e0e0ff;margin:0'>🏥 Early Sepsis Detection System</h2>
</div>
"""))

upload_btn = widgets.FileUpload(accept='.psv,.csv', multiple=False)
analyse_btn = widgets.Button(description='⚡ Analyse Patient', button_style='danger')
output = widgets.Output()

display(upload_btn, analyse_btn, output)


# ======================
# ANALYSIS FUNCTION
# ======================
def on_analyse(b):
    with output:
        clear_output()

        if not upload_btn.value:
            print("No file uploaded")
            return

        fname   = list(upload_btn.value.keys())[0]
        content = upload_btn.value[fname]['content']

        # load file
        try:
            df = pd.read_csv(io.BytesIO(content), sep='|')
        except:
            df = pd.read_csv(io.BytesIO(content))

        print("Loaded:", fname)

        try:
            # 🔥 FIX 1: normalize columns
            df = normalize_columns(df)

            # 🔥 FIX 2: extract features
            row_dict = extract_features(df)
            row_df = pd.DataFrame([row_dict])

            # 🔥 FIX 3: match training features
            for f in features:
                if f not in row_df.columns:
                    row_df[f] = 0.0

            row_df = row_df[features]

            # 🔥 FIX 4: predict
            prob = model.predict_proba(row_df.values)[0][1]

        except Exception as e:
            print("Prediction error:", e)
            return

        print("\nSepsis Probability:", round(prob, 4))

        if prob > threshold:
            print("🚨 SEPSIS DETECTED")
        else:
            print("✅ No Sepsis")


analyse_btn.on_click(on_analyse)